In [29]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


BASE = "https://www.scout.org"
PAST_EVENTS_URL = (
    "https://www.scout.org/events-search"
    "?field_event_date_value_op=%3C"
    "&field_event_date_value%5Bvalue%5D=now"
)
OUTPUT_XLSX = "wosm_past_events.xlsx"

HEADLESS = False          # keep False initially (anti-bot/consent)
WAIT_SEC = 60
MAX_SHOW_MORE_CLICKS = 300
DELAY_BETWEEN_PAGES = 0.25


def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def setup_driver() -> webdriver.Chrome:
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)


def wait_for_cards(driver, wait: WebDriverWait):
    # Your snippet shows qz-card elements in the DOM.
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "qz-card[url]")) > 0)


def click_show_more_until_done(driver):
    """
    Click 'Show more events' until it disappears or stops increasing qz-card count.
    """
    stagnant = 0

    for _ in range(MAX_SHOW_MORE_CLICKS):
        buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
        if not buttons:
            break

        before = len(driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"))

        driver.execute_script("arguments[0].scrollIntoView({block:'center'});", buttons[0])
        time.sleep(0.6)

        try:
            buttons[0].click()
        except Exception:
            driver.execute_script("arguments[0].click();", buttons[0])

        # allow ajax to append
        time.sleep(1.4)

        after = len(driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"))
        if after <= before:
            stagnant += 1
            if stagnant >= 3:
                break
        else:
            stagnant = 0


def collect_event_urls_from_cards(driver) -> list[str]:
    """
    qz-card has url="/event/slug" so we can get all event URLs reliably.
    """
    urls = []
    seen = set()

    for card in driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"):
        href = card.get_attribute("url") or ""
        full = urljoin(BASE, href)
        key = full.rstrip("/")
        if "/event/" not in full:
            continue
        if key in seen:
            continue
        seen.add(key)
        urls.append(full)

    return urls


# -------- detail page extraction (sidebar) --------
def get_label_block(soup: BeautifulSoup, label: str):
    for block in soup.select("aside .mb-6"):
        head = block.select_one(".font-semibold")
        if head and norm_text(head.get_text()) == label:
            return block
    return None


def extract_first_value_after_label(soup: BeautifulSoup, label: str) -> str:
    block = get_label_block(soup, label)
    if not block:
        return ""
    divs = block.find_all("div", recursive=False)
    lines = [norm_text(d.get_text()) for d in divs if norm_text(d.get_text())]
    for t in lines:
        if t != label:
            return t
    return ""


def extract_location(soup: BeautifulSoup) -> str:
    block = get_label_block(soup, "Location")
    if not block:
        return ""
    divs = block.find_all("div", recursive=False)
    parts = []
    for d in divs:
        t = norm_text(d.get_text())
        if not t or t == "Location":
            continue
        parts.append(t)
    return ", ".join([p for p in parts if p])


def extract_organizers(soup: BeautifulSoup) -> str:
    block = get_label_block(soup, "Organizers")
    if not block:
        return ""
    divs = block.find_all("div", recursive=False)
    vals = []
    for d in divs:
        t = norm_text(d.get_text())
        if not t or t == "Organizers":
            continue
        vals.append(t)
    return ", ".join(dict.fromkeys(vals))


def extract_sdgs(soup: BeautifulSoup) -> str:
    for block in soup.select("aside .mb-6"):
        head = block.select_one(".font-semibold")
        if not head:
            continue
        if norm_text(head.get_text()).startswith("SDGs"):
            vals = []
            for el in block.select(".text-theme-violet .my-2, .text-theme-violet div"):
                t = norm_text(el.get_text())
                if t:
                    vals.append(t)
            return ", ".join(dict.fromkeys(vals))
    return ""


def parse_event_detail(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    title_el = soup.select_one("h1")
    event_name = norm_text(title_el.get_text()) if title_el else ""

    start_date = extract_first_value_after_label(soup, "Started on")
    end_date = extract_first_value_after_label(soup, "Ended on")
    location = extract_location(soup)
    organizers = extract_organizers(soup)
    sdg = extract_sdgs(soup)

    return {
        "event_name": event_name,
        "start_date": start_date,
        "end_date": end_date,
        "location": location,
        "organizers": organizers,
        "sdg": sdg,
        "url": url,
    }


def scrape_all_past_events() -> pd.DataFrame:
    driver = setup_driver()
    wait = WebDriverWait(driver, WAIT_SEC)

    try:
        driver.get(PAST_EVENTS_URL)
        time.sleep(3)

        # if the site shows consent/antibot, solve it once manually now
        wait_for_cards(driver, wait)

        click_show_more_until_done(driver)

        event_urls = collect_event_urls_from_cards(driver)
        print(f"Collected {len(event_urls)} event URLs")

        rows = []
        for i, url in enumerate(event_urls, start=1):
            print(f"[{i}/{len(event_urls)}] {url}")
            driver.get(url)

            WebDriverWait(driver, WAIT_SEC).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, "aside")) > 0 or len(d.find_elements(By.CSS_SELECTOR, "h1")) > 0
            )

            rows.append(parse_event_detail(driver.page_source, url))
            time.sleep(DELAY_BETWEEN_PAGES)

        df = pd.DataFrame(rows)
        if df.empty:
            return df

        return df[["event_name", "start_date", "end_date", "location", "organizers", "sdg"]]

    finally:
        driver.quit()


def save_to_csv(df, path="wosm_past_events.csv"):
    if df is None or df.empty:
        print("No events found; nothing to save.")
        return
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} rows to {path}")


if __name__ == "__main__":
    # pip install -U selenium webdriver-manager pandas openpyxl beautifulsoup4
    df = scrape_all_past_events()
    print("Rows:", len(df))
    print(df.head(10))
    save_to_excel(df)

Collected 67 event URLs
[1/67] https://www.scout.org/event/4th-eurasia-youth-forum
[2/67] https://www.scout.org/event/7th-eurasia-scout-conference
[3/67] https://www.scout.org/event/nices-2021
[4/67] https://www.scout.org/event/26th-australian-jamboree-aj2022
[5/67] https://www.scout.org/event/hile-2022
[6/67] https://www.scout.org/event/finnjamboree-kajo-2022
[7/67] https://www.scout.org/event/jamboree-denmark-2022
[8/67] https://www.scout.org/event/vcp-national-jamboree
[9/67] https://www.scout.org/event/elpida-hellenic-national-jamboree-2022
[10/67] https://www.scout.org/event/youth-advocate-training-europe
[11/67] https://www.scout.org/event/jota-joti-2022
[12/67] https://www.scout.org/event/european-sustainability-forum-2022
[13/67] https://www.scout.org/event/european-communications-forum
[14/67] https://www.scout.org/event/academy-2022
[15/67] https://www.scout.org/event/azimuth-platform-line-training
[16/67] https://www.scout.org/event/scouts-humanitarians-youth-panel
[17/67] h

Exception ignored in: <function ZipFile.__del__ at 0x00000213BE31CEA0>
Traceback (most recent call last):
  File "C:\Users\brahm\anaconda3\Lib\zipfile\__init__.py", line 1988, in __del__
    self.close()
  File "C:\Users\brahm\anaconda3\Lib\zipfile\__init__.py", line 2005, in close
    self.fp.seek(self.start_dir)
ValueError: seek of closed file


[20/67] https://www.scout.org/event/european-youth-agora
[21/67] https://www.scout.org/event/1st-africa-rover-moot/kenya-2023
[22/67] https://www.scout.org/event/advocates-europe-cancelled
[23/67] https://www.scout.org/event/apr-awareness-workshop-wellness-and-safety-scouting
[24/67] https://www.scout.org/event/overture-diversity-network
[25/67] https://www.scout.org/event/22nd-apr-scout-photo-contest
[26/67] https://www.scout.org/event/apr-partnerships-forum
[27/67] https://www.scout.org/event/weact-forum
[28/67] https://www.scout.org/event/asia-pacific-youth-mobilisation-summit
[29/67] https://www.scout.org/event/25th-world-scout-jamboree-korea-2023
[30/67] https://www.scout.org/event/10th-european-educational-methods-forum
[31/67] https://www.scout.org/event/apr-humanitarian-action-workshop
[32/67] https://www.scout.org/event/academy-2023
[33/67] https://www.scout.org/event/european-networking-event
[34/67] https://www.scout.org/event/chief-commissioners-network-meeting-and-chief
[3

TypeError: XlsxWriter._write_cells() got an unexpected keyword argument 'autofilter_range'

In [31]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager


BASE = "https://www.scout.org"
PAST_EVENTS_URL = (
    "https://www.scout.org/events-search"
    "?field_event_date_value_op=%3C"
    "&field_event_date_value%5Bvalue%5D=now"
)
OUTPUT_CSV = "wosm_past_events.csv"

HEADLESS = False
WAIT_SEC = 60
MAX_SHOW_MORE_CLICKS = 300
DELAY_BETWEEN_PAGES = 0.25


def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def setup_driver() -> webdriver.Chrome:
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)


def wait_for_cards(driver, wait: WebDriverWait):
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "qz-card[url]")) > 0)


def click_show_more_until_done(driver):
    stagnant = 0

    for _ in range(MAX_SHOW_MORE_CLICKS):
        try:
            buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
            if not buttons:
                break

            before = len(driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"))

            # Re-find just before clicking to avoid stale refs
            buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
            if not buttons:
                break

            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", buttons[0])
            time.sleep(0.6)

            try:
                buttons[0].click()
            except StaleElementReferenceException:
                # re-find and try again
                buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
                if not buttons:
                    break
                driver.execute_script("arguments[0].click();", buttons[0])
            except Exception:
                driver.execute_script("arguments[0].click();", buttons[0])

            time.sleep(1.4)

            after = len(driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"))
            if after <= before:
                stagnant += 1
                if stagnant >= 3:
                    break
            else:
                stagnant = 0

        except StaleElementReferenceException:
            # DOM updated between find & click, just retry loop
            continue


def collect_event_urls_from_cards(driver) -> list[str]:
    urls = []
    seen = set()

    for card in driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"):
        href = card.get_attribute("url") or ""
        full = urljoin(BASE, href)
        key = full.rstrip("/")
        if "/event/" not in full:
            continue
        if key in seen:
            continue
        seen.add(key)
        urls.append(full)

    return urls


def get_label_block(soup: BeautifulSoup, label: str):
    for block in soup.select("aside .mb-6"):
        head = block.select_one(".font-semibold")
        if head and norm_text(head.get_text()) == label:
            return block
    return None


def extract_first_value_after_label(soup: BeautifulSoup, label: str) -> str:
    block = get_label_block(soup, label)
    if not block:
        return ""
    divs = block.find_all("div", recursive=False)
    lines = [norm_text(d.get_text()) for d in divs if norm_text(d.get_text())]
    for t in lines:
        if t != label:
            return t
    return ""


def extract_location(soup: BeautifulSoup) -> str:
    block = get_label_block(soup, "Location")
    if not block:
        return ""
    divs = block.find_all("div", recursive=False)
    parts = []
    for d in divs:
        t = norm_text(d.get_text())
        if not t or t == "Location":
            continue
        parts.append(t)
    return ", ".join([p for p in parts if p])


def extract_organizers(soup: BeautifulSoup) -> str:
    block = get_label_block(soup, "Organizers")
    if not block:
        return ""
    divs = block.find_all("div", recursive=False)
    vals = []
    for d in divs:
        t = norm_text(d.get_text())
        if not t or t == "Organizers":
            continue
        vals.append(t)
    return ", ".join(dict.fromkeys(vals))


def extract_sdgs(soup: BeautifulSoup) -> str:
    for block in soup.select("aside .mb-6"):
        head = block.select_one(".font-semibold")
        if not head:
            continue
        if norm_text(head.get_text()).startswith("SDGs"):
            vals = []
            for el in block.select(".text-theme-violet .my-2, .text-theme-violet div"):
                t = norm_text(el.get_text())
                if t:
                    vals.append(t)
            return ", ".join(dict.fromkeys(vals))
    return ""


def parse_event_detail(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    title_el = soup.select_one("h1")
    event_name = norm_text(title_el.get_text()) if title_el else ""

    start_date = extract_first_value_after_label(soup, "Started on")
    end_date = extract_first_value_after_label(soup, "Ended on")
    location = extract_location(soup)
    organizers = extract_organizers(soup)
    sdg = extract_sdgs(soup)

    return {
        "event_name": event_name,
        "start_date": start_date,
        "end_date": end_date,
        "location": location,
        "organizers": organizers,
        "sdg": sdg,
        "url": url,
    }


def scrape_all_past_events() -> pd.DataFrame:
    driver = setup_driver()
    wait = WebDriverWait(driver, WAIT_SEC)

    try:
        driver.get(PAST_EVENTS_URL)
        time.sleep(3)

        wait_for_cards(driver, wait)

        click_show_more_until_done(driver)

        event_urls = collect_event_urls_from_cards(driver)
        print(f"Collected {len(event_urls)} event URLs")

        rows = []
        for i, url in enumerate(event_urls, start=1):
            print(f"[{i}/{len(event_urls)}] {url}")
            driver.get(url)

            WebDriverWait(driver, WAIT_SEC).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, "aside")) > 0
                or len(d.find_elements(By.CSS_SELECTOR, "h1")) > 0
            )

            rows.append(parse_event_detail(driver.page_source, url))
            time.sleep(DELAY_BETWEEN_PAGES)

        df = pd.DataFrame(rows)
        if df.empty:
            return df

        return df[["event_name", "start_date", "end_date", "location", "organizers", "sdg"]]

    finally:
        driver.quit()


def save_to_csv(df, path=OUTPUT_CSV):
    if df is None or df.empty:
        print("No events found; nothing to save.")
        return
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(df)} rows to {path}")


if __name__ == "__main__":
    df = scrape_all_past_events()
    print("Rows:", len(df))
    print(df.head(10))
    save_to_csv(df)

Collected 67 event URLs
[1/67] https://www.scout.org/event/4th-eurasia-youth-forum
[2/67] https://www.scout.org/event/7th-eurasia-scout-conference
[3/67] https://www.scout.org/event/nices-2021
[4/67] https://www.scout.org/event/26th-australian-jamboree-aj2022
[5/67] https://www.scout.org/event/hile-2022
[6/67] https://www.scout.org/event/finnjamboree-kajo-2022
[7/67] https://www.scout.org/event/jamboree-denmark-2022
[8/67] https://www.scout.org/event/vcp-national-jamboree
[9/67] https://www.scout.org/event/elpida-hellenic-national-jamboree-2022
[10/67] https://www.scout.org/event/youth-advocate-training-europe
[11/67] https://www.scout.org/event/jota-joti-2022
[12/67] https://www.scout.org/event/european-sustainability-forum-2022
[13/67] https://www.scout.org/event/european-communications-forum
[14/67] https://www.scout.org/event/academy-2022
[15/67] https://www.scout.org/event/azimuth-platform-line-training
[16/67] https://www.scout.org/event/scouts-humanitarians-youth-panel
[17/67] h

In [32]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

from openpyxl import Workbook


BASE = "https://www.scout.org"
PAST_EVENTS_URL = (
    "https://www.scout.org/events-search"
    "?field_event_date_value_op=%3C"
    "&field_event_date_value%5Bvalue%5D=now"
)
OUTPUT_XLSX = "wosm_past_events.xlsx"

HEADLESS = False
WAIT_SEC = 60
MAX_SHOW_MORE_CLICKS = 300
DELAY_BETWEEN_PAGES = 0.25


def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def setup_driver() -> webdriver.Chrome:
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)


def wait_for_cards(driver, wait: WebDriverWait):
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "qz-card[url]")) > 0)


def click_show_more_until_done(driver):
    stagnant = 0

    for _ in range(MAX_SHOW_MORE_CLICKS):
        try:
            buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
            if not buttons:
                break

            before = len(driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"))

            # re-find button just before clicking
            buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
            if not buttons:
                break

            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", buttons[0])
            time.sleep(0.6)

            try:
                buttons[0].click()
            except StaleElementReferenceException:
                buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
                if not buttons:
                    break
                driver.execute_script("arguments[0].click();", buttons[0])
            except Exception:
                driver.execute_script("arguments[0].click();", buttons[0])

            time.sleep(1.4)

            after = len(driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"))
            if after <= before:
                stagnant += 1
                if stagnant >= 3:
                    break
            else:
                stagnant = 0

        except StaleElementReferenceException:
            continue


def collect_event_urls_from_cards(driver) -> list[str]:
    urls = []
    seen = set()

    for card in driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"):
        href = card.get_attribute("url") or ""
        full = urljoin(BASE, href)
        key = full.rstrip("/")
        if "/event/" not in full:
            continue
        if key in seen:
            continue
        seen.add(key)
        urls.append(full)

    return urls


def get_label_block(soup: BeautifulSoup, label: str):
    for block in soup.select("aside .mb-6"):
        head = block.select_one(".font-semibold")
        if head and norm_text(head.get_text()) == label:
            return block
    return None


def extract_first_value_after_label(soup: BeautifulSoup, label: str) -> str:
    block = get_label_block(soup, label)
    if not block:
        return ""
    divs = block.find_all("div", recursive=False)
    lines = [norm_text(d.get_text()) for d in divs if norm_text(d.get_text())]
    for t in lines:
        if t != label:
            return t
    return ""


def extract_location(soup: BeautifulSoup) -> str:
    block = get_label_block(soup, "Location")
    if not block:
        return ""
    divs = block.find_all("div", recursive=False)
    parts = []
    for d in divs:
        t = norm_text(d.get_text())
        if not t or t == "Location":
            continue
        parts.append(t)
    return ", ".join([p for p in parts if p])


def extract_organizers(soup: BeautifulSoup) -> str:
    block = get_label_block(soup, "Organizers")
    if not block:
        return ""
    divs = block.find_all("div", recursive=False)
    vals = []
    for d in divs:
        t = norm_text(d.get_text())
        if not t or t == "Organizers":
            continue
        vals.append(t)
    return ", ".join(dict.fromkeys(vals))


def extract_sdgs(soup: BeautifulSoup) -> str:
    for block in soup.select("aside .mb-6"):
        head = block.select_one(".font-semibold")
        if not head:
            continue
        if norm_text(head.get_text()).startswith("SDGs"):
            vals = []
            for el in block.select(".text-theme-violet .my-2, .text-theme-violet div"):
                t = norm_text(el.get_text())
                if t:
                    vals.append(t)
            return ", ".join(dict.fromkeys(vals))
    return ""


def parse_event_detail(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    title_el = soup.select_one("h1")
    event_name = norm_text(title_el.get_text()) if title_el else ""

    start_date = extract_first_value_after_label(soup, "Started on")
    end_date = extract_first_value_after_label(soup, "Ended on")
    location = extract_location(soup)
    organizers = extract_organizers(soup)
    sdg = extract_sdgs(soup)

    return {
        "event_name": event_name,
        "start_date": start_date,
        "end_date": end_date,
        "location": location,
        "organizers": organizers,
        "sdg": sdg,
        "url": url,
    }


def scrape_all_past_events() -> pd.DataFrame:
    driver = setup_driver()
    wait = WebDriverWait(driver, WAIT_SEC)

    try:
        driver.get(PAST_EVENTS_URL)
        time.sleep(3)

        wait_for_cards(driver, wait)
        click_show_more_until_done(driver)

        event_urls = collect_event_urls_from_cards(driver)
        print(f"Collected {len(event_urls)} event URLs")

        rows = []
        for i, url in enumerate(event_urls, start=1):
            print(f"[{i}/{len(event_urls)}] {url}")
            driver.get(url)

            WebDriverWait(driver, WAIT_SEC).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, "aside")) > 0
                or len(d.find_elements(By.CSS_SELECTOR, "h1")) > 0
            )

            rows.append(parse_event_detail(driver.page_source, url))
            time.sleep(DELAY_BETWEEN_PAGES)

        df = pd.DataFrame(rows)
        if df.empty:
            return df

        return df[["event_name", "start_date", "end_date", "location", "organizers", "sdg"]]

    finally:
        driver.quit()


def save_to_excel_manual(df, path=OUTPUT_XLSX):
    if df is None or df.empty:
        print("No events found; nothing to save.")
        return

    wb = Workbook()
    ws = wb.active
    ws.title = "Events"

    # header
    ws.append(list(df.columns))

    # rows
    for row in df.itertuples(index=False):
        ws.append(list(row))

    wb.save(path)
    print(f"Saved {len(df)} rows to {path}")


if __name__ == "__main__":
    df = scrape_all_past_events()
    print("Rows:", len(df))
    print(df.head(10))
    save_to_excel_manual(df)

Collected 67 event URLs
[1/67] https://www.scout.org/event/4th-eurasia-youth-forum
[2/67] https://www.scout.org/event/7th-eurasia-scout-conference
[3/67] https://www.scout.org/event/nices-2021
[4/67] https://www.scout.org/event/26th-australian-jamboree-aj2022
[5/67] https://www.scout.org/event/hile-2022
[6/67] https://www.scout.org/event/finnjamboree-kajo-2022
[7/67] https://www.scout.org/event/jamboree-denmark-2022
[8/67] https://www.scout.org/event/vcp-national-jamboree
[9/67] https://www.scout.org/event/elpida-hellenic-national-jamboree-2022
[10/67] https://www.scout.org/event/youth-advocate-training-europe
[11/67] https://www.scout.org/event/jota-joti-2022
[12/67] https://www.scout.org/event/european-sustainability-forum-2022
[13/67] https://www.scout.org/event/european-communications-forum
[14/67] https://www.scout.org/event/academy-2022
[15/67] https://www.scout.org/event/azimuth-platform-line-training
[16/67] https://www.scout.org/event/scouts-humanitarians-youth-panel
[17/67] h

In [35]:
import re
import time
from urllib.parse import urljoin

import pandas as pd
from bs4 import BeautifulSoup

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import StaleElementReferenceException
from webdriver_manager.chrome import ChromeDriverManager

from openpyxl import Workbook


BASE = "https://www.scout.org"
NEWS_URL = "https://www.scout.org/news"
OUTPUT_XLSX = "wosm_news_details.xlsx"

HEADLESS = False
WAIT_SEC = 60
MAX_SHOW_MORE_CLICKS = 300
DELAY_BETWEEN_PAGES = 0.3


def norm_text(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()


def setup_driver() -> webdriver.Chrome:
    opts = Options()
    if HEADLESS:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")

    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=opts)


def wait_for_cards(driver, wait: WebDriverWait):
    wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "qz-card, article, .views-row")) > 0)


def click_show_more_until_done(driver):
    stagnant = 0

    for _ in range(MAX_SHOW_MORE_CLICKS):
        try:
            buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
            if not buttons:
                break

            before = len(driver.find_elements(By.CSS_SELECTOR, "qz-card, article, .views-row"))

            buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
            if not buttons:
                break

            driver.execute_script("arguments[0].scrollIntoView({block:'center'});", buttons[0])
            time.sleep(0.6)

            try:
                buttons[0].click()
            except StaleElementReferenceException:
                buttons = driver.find_elements(By.CSS_SELECTOR, "a.button[rel='next'], a.button[title='Load more items']")
                if not buttons:
                    break
                driver.execute_script("arguments[0].click();", buttons[0])
            except Exception:
                driver.execute_script("arguments[0].click();", buttons[0])

            time.sleep(1.4)

            after = len(driver.find_elements(By.CSS_SELECTOR, "qz-card, article, .views-row"))
            if after <= before:
                stagnant += 1
                if stagnant >= 3:
                    break
            else:
                stagnant = 0

        except StaleElementReferenceException:
            continue


def collect_article_urls(driver) -> list[str]:
    """
    Collect URLs from card components.
    """
    urls = []
    seen = set()

    # Prefer qz-card[url]
    for card in driver.find_elements(By.CSS_SELECTOR, "qz-card[url]"):
        href = card.get_attribute("url") or ""
        full = urljoin(BASE, href)
        key = full.rstrip("/")
        if "/news/" not in full:
            continue
        if key in seen:
            continue
        seen.add(key)
        urls.append(full)

    # Fallback: any link to /news/
    if not urls:
        for a in driver.find_elements(By.CSS_SELECTOR, "a[href*='/news/']"):
            href = a.get_attribute("href") or ""
            if "/news/" not in href:
                continue
            key = href.rstrip("/")
            if key in seen:
                continue
            seen.add(key)
            urls.append(href)

    return urls


# -------- detail page extraction --------
def extract_detail_fields(html: str, url: str) -> dict:
    soup = BeautifulSoup(html, "html.parser")

    # Title
    title_el = soup.select_one("h1")
    title = norm_text(title_el.get_text()) if title_el else ""

    # Date (try time tag)
    date = ""
    time_el = soup.select_one("time")
    if time_el:
        date = norm_text(time_el.get_text())

    # Category/type (common label styles)
    category = ""
    cat_el = soup.select_one(".meta .font-bold, .meta strong, .field--name-field-news-type, .field--name-field-category")
    if cat_el:
        category = norm_text(cat_el.get_text())

    # Location (if present in sidebar or fields)
    location = ""
    loc_el = soup.select_one(".field--name-field-location, .field--name-field-event-location, .location, aside .mb-6")
    if loc_el:
        # If it's a labeled block, strip label
        text = norm_text(loc_el.get_text())
        text = re.sub(r"^Location\s*", "", text)
        location = text

    return {
        "title": title,
        "date": date,
        "category_type": category,
        "location": location,
        "url": url,
    }


def scrape_news_details() -> pd.DataFrame:
    driver = setup_driver()
    wait = WebDriverWait(driver, WAIT_SEC)

    try:
        driver.get(NEWS_URL)
        time.sleep(3)

        wait_for_cards(driver, wait)
        click_show_more_until_done(driver)

        urls = collect_article_urls(driver)
        print(f"Collected {len(urls)} news URLs")

        rows = []
        for i, url in enumerate(urls, start=1):
            print(f"[{i}/{len(urls)}] {url}")
            driver.get(url)

            WebDriverWait(driver, WAIT_SEC).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, "h1")) > 0
            )

            rows.append(extract_detail_fields(driver.page_source, url))
            time.sleep(DELAY_BETWEEN_PAGES)

        df = pd.DataFrame(rows)
        if df.empty:
            return df

        return df[["title", "date", "category_type", "location", "url"]]

    finally:
        driver.quit()


def save_to_excel_manual(df, path=OUTPUT_XLSX):
    if df is None or df.empty:
        print("No news found; nothing to save.")
        return

    wb = Workbook()
    ws = wb.active
    ws.title = "News"

    ws.append(list(df.columns))
    for row in df.itertuples(index=False):
        ws.append(list(row))

    wb.save(path)
    print(f"Saved {len(df)} rows to {path}")


if __name__ == "__main__":
    df = scrape_news_details()
    print("Rows:", len(df))
    print(df.head(10))
    save_to_excel_manual(df)

Collected 73 news URLs
[1/73] https://www.scout.org/news/story/scouting-learners-and-educators-co-creating
[2/73] https://www.scout.org/news/gambia-scout-association-revamps-youth-programme
[3/73] https://www.scout.org/news/news/goran-gjorgjiev-appointed-regional-director
[4/73] https://www.scout.org/news/jamcam-2025-over-2/000-young-scouts-gather-cali
[5/73] https://www.scout.org/news/news/3rd-arab-parajamboree-celebrates-accessibility
[6/73] https://www.scout.org/news/thousands-gather-33rd-asia-pacific-regional-scout
[7/73] https://www.scout.org/news/news/international-leadership-training-brings-together
[8/73] https://www.scout.org/news/news/scouts-learn-about-impact-innovation-and-creative
[9/73] https://www.scout.org/news/green-skills-youth-closing-story-growth-and-community
[10/73] https://www.scout.org/news/how-bojana-discovered-small-actions-can-create-big
[11/73] https://www.scout.org/news/two-steps-leadership/one-step-closer-change
[12/73] https://www.scout.org/news/croatian-